# D6 — multi-head joint ablation (one compound at a time)

Spec (approved 2026-06-29): `docs/superpowers/specs/2026-06-29-multihead-lexical-ablation-design.md`.
Pre-registration: DECISIONS.md D6 entry — **commit before the first real forward pass**.
Claim under test, scoped: even the strongest lexical lead (screen reader) is
distributed across heads, not a localized circuit. Model: pythia-2.8b throughout.

**Upload to this Colab session:** this notebook + the `src/` folder (drag the folder
into the Files sidebar) + `results/pythia/pythia-2.8b-binding.csv` (into
`results/pythia/`, so Stage A reads the frozen sweep; out-of-sweep compounds like
stock_market run a fresh in-memory binding pass and say so in the ledger).

Flow: run Setup once → `sanity_check` once → edit Cell 1 (compound) → run Cell 2 →
repeat for screen_reader, alt_text, stock_market, semantic_html → zip and bring it
home. Model loads once per session; only the compound changes between visits.

## Setup

In [ ]:
# Cell 00: Colab Stuff
import os

try:
    import google.colab
    IN_COLAB = True
    print("Running as a Colab notebook")
    os.system('pip install transformer_lens==2.17.0 --quiet')
    os.system('pip install transformers==4.57.6 --quiet')
    # NOTE: no 'pip install --upgrade numpy' here — on current Colab images it
    # desyncs numpy from the preinstalled scipy's compiled ABI
    # (ImportError: _center from numpy._core.umath). D8's Cell 00 still carries
    # the line above its own removal note; fixed here, fix D8 when convenient.
    print("Dependencies installed")
except ImportError:
    IN_COLAB = False

In [ ]:
# Cell 0: Project root + imports (expects the src/ FOLDER uploaded, not flat files)
import sys
from pathlib import Path
import torch

# Colab: files are in /content/ ; Local: notebook lives in notebooks/
if Path('/content/src').exists():
    PROJECT_ROOT = Path('/content')
else:
    PROJECT_ROOT = Path.cwd().parent

assert (PROJECT_ROOT / 'src' / 'd6_multihead_ablation.py').exists(), (
    'Upload the src/ folder next to this notebook (Files sidebar, drag the '
    'whole folder) — d6 needs binding, head_characterization, qk_ov, and '
    'd6_multihead_ablation together.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.d6_multihead_ablation import run_d6, sanity_check, COMPOUNDS
print(f"Project root: {PROJECT_ROOT}")
print("Compounds in the panel:", list(COMPOUNDS))

In [ ]:
# Cell 0b: Load the model ONCE per session (single model, per spec)
from transformer_lens import HookedTransformer

if 'model' not in globals():
    device = ('cuda' if torch.cuda.is_available()
              else 'mps' if torch.backends.mps.is_available() else 'cpu')
    model = HookedTransformer.from_pretrained('pythia-2.8b', device=device)
    print(f"Loaded pythia-2.8b on {device} "
          f"({model.cfg.n_layers} layers x {model.cfg.n_heads} heads)")
else:
    print("Model already loaded — reusing.")

In [ ]:
# Cell 0c: Sanity asserts (spec: identity KL==0; plural==singular on L29/H7;
# late-layer set moves KL). Run ONCE before the first real ablation.
sanity_check(model)

## One compound per visit
Edit Cell 1, run Cell 2. Order: `screen_reader` (primary — also runs the
bicycle-wheel negative control) → `alt_text` → `stock_market` → `semantic_html`.
Cell 3 shows the ledger. Reruns replace a compound's rows, never duplicate.

In [ ]:
# Cell 1: Compound name variable
# screen_reader | alt_text | stock_market | semantic_html
compound_name = "screen_reader"

In [ ]:
# Cell 2: Run D6 for this compound
# Stage A earns the lexical set (deep + selective + not sink/structural; the
# excluded heads become the labeled positive-control tail), Stage B runs the
# cumulative-knockout curve, both ledgers upsert. Seal untouched, no generation.
curves = run_d6(model, PROJECT_ROOT, compound_name)
curves

In [ ]:
# Cell 3: Ledger peek — which compounds exist so far, and the verdict shape
from pathlib import Path
import pandas as pd
led = PROJECT_ROOT / 'results/pythia/pythia-2.8b-multihead-ablation.csv'
if led.exists():
    df = pd.read_csv(led)
    print(df.groupby(['compound', 'role', 'segment'])['kl']
            .agg(['count', 'max']).round(6))
else:
    print('no ledger yet')
cand = PROJECT_ROOT / 'results/pythia/pythia-2.8b-candidate-heads.csv'
if cand.exists():
    cdf = pd.read_csv(cand)
    print('\nearned lexical sets:',
          cdf[cdf['in_lexical_set']].groupby('compound')
             .apply(lambda g: ', '.join(f"L{r.layer}H{r.head}"
                                        for r in g.itertuples()),
                    include_groups=False).to_dict())

## Bring it home

In [ ]:
# Cell 4: Zip + download (loss class: ephemeral /content)
import os
os.system('zip -j d6_results.zip '
          'results/pythia/pythia-2.8b-candidate-heads.csv '
          'results/pythia/pythia-2.8b-multihead-ablation.csv')
if IN_COLAB:
    from google.colab import files
    files.download('d6_results.zip')
else:
    print('Local run — results already live in results/pythia/, no zip needed.')